# Response Clarity Classification with Qwen Prompting

**Student ID:** sdi2200160  
**Course:** Artificial Intelligence II - Deep Learning for NLP  
**Assignment:** Homework 3 - Prompting with Qwen-family language models

This notebook is self-contained for Kaggle submission. It loads the CLARITY/QEvasion dataset, runs Qwen prompting experiments on a stratified validation sample, selects the best model/strategy by validation **macro F1**, and writes the required Kaggle submission file:

```text
submission_best_prompting_system.csv
```

Macro F1 is the primary selection criterion because the assignment is a three-class classification task and the `Ambivalent` class is the central failure mode. Accuracy is still reported and is used only as a tie-breaker. The notebook also saves intermediate validation generations and summary tables under `runs_hw3_kaggle/` so the selected system is traceable.


## Runtime Checks

The notebook follows the same Kaggle pattern used in the previous homeworks:

- enable a GPU runtime before running prompted inference;
- keep Internet enabled for the first run so Hugging Face can download the dataset and Qwen weights;
- the first cell upgrades the Hugging Face runtime stack on Kaggle, and installs the current Transformers source build when needed for the `qwen3_5` architecture;
- if the dataset or models are gated in your environment, add an `HF_TOKEN` Kaggle secret;
- the final CSV is written to the current working directory, which is `/kaggle/working` on Kaggle.

The validation comparison is intentionally small by default (`10` examples per class) because prompted generation has no training phase but is expensive at inference time. The final submission is always generated over the full official test split.


In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import os
from pathlib import Path
import platform
import subprocess
import sys

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle').exists()
FORCE_MODERN_HF_RUNTIME = IS_KAGGLE
FORCE_TRANSFORMERS_SOURCE = IS_KAGGLE
REQUIRED_RUNTIME = {
    'datasets': 'datasets>=4.0.0',
    'accelerate': 'accelerate>=1.0.0',
    'sentencepiece': 'sentencepiece>=0.2.0',
    'huggingface_hub': 'huggingface_hub>=0.30.0',
    'safetensors': 'safetensors>=0.4.5',
    'tiktoken': 'tiktoken>=0.7.0',
}
TRANSFORMERS_PYPI_SPEC = 'transformers>=4.52.0'
TRANSFORMERS_SOURCE_SPECS = (
    'git+https://github.com/huggingface/transformers.git',
    'https://github.com/huggingface/transformers/archive/refs/heads/main.zip',
)
QWEN_MODEL_TYPE = 'qwen3_5'
QWEN_CONFIG_PROBE_MODEL = 'Qwen/Qwen3.5-0.8B'
CORE_MODULES = ('numpy', 'pandas', 'sklearn', 'torch')


def module_exists(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def purge_imported_package(package_name: str) -> None:
    for module_name in list(sys.modules):
        if module_name == package_name or module_name.startswith(package_name + '.'):
            del sys.modules[module_name]


def pip_install(pip_spec: str) -> None:
    print(f'[install] {pip_spec}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pip_spec])
    importlib.invalidate_caches()


def ensure_package(module_name: str, pip_spec: str, force_upgrade: bool = False) -> None:
    if module_exists(module_name) and not force_upgrade:
        print(f'[ok] {module_name} already available')
        return

    try:
        pip_install(pip_spec)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            f'Failed to install {pip_spec}. If this is Kaggle, enable Internet for the first run.'
        ) from error

    purge_imported_package(module_name)
    if not module_exists(module_name):
        raise RuntimeError(f'Installed {pip_spec}, but {module_name} is still unavailable.')


def transformers_supports_qwen35() -> bool:
    if not module_exists('transformers'):
        return False
    try:
        import transformers as transformers_probe
        from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    except Exception as error:
        print(f'[needs-install] could not inspect transformers: {error}')
        return False

    if QWEN_MODEL_TYPE in CONFIG_MAPPING_NAMES:
        print(f'[ok] transformers {transformers_probe.__version__} recognizes {QWEN_MODEL_TYPE}')
        return True

    try:
        transformers_probe.AutoConfig.from_pretrained(QWEN_CONFIG_PROBE_MODEL, trust_remote_code=True)
    except Exception as error:
        print(f'[needs-install] transformers {transformers_probe.__version__} cannot load {QWEN_CONFIG_PROBE_MODEL}: {error}')
        return False

    print(f'[ok] transformers {transformers_probe.__version__} loads {QWEN_CONFIG_PROBE_MODEL} through remote code')
    return True


def install_transformers_from_source() -> bool:
    for source_spec in TRANSFORMERS_SOURCE_SPECS:
        try:
            pip_install(source_spec)
        except subprocess.CalledProcessError as error:
            print(f'[warn] failed to install {source_spec}: {error}')
            continue

        purge_imported_package('transformers')
        if transformers_supports_qwen35():
            return True
    return False


def ensure_transformers_for_qwen35() -> None:
    if FORCE_TRANSFORMERS_SOURCE:
        print('[info] Kaggle runtime detected; installing Transformers from source for Qwen3.5 support')
        if install_transformers_from_source():
            return

    ensure_package('transformers', TRANSFORMERS_PYPI_SPEC, force_upgrade=FORCE_MODERN_HF_RUNTIME)
    if transformers_supports_qwen35():
        return

    if install_transformers_from_source():
        return

    raise RuntimeError(
        'Transformers is installed, but it still does not recognize the Qwen3.5 model type '
        f'`{QWEN_MODEL_TYPE}`. Enable Internet on Kaggle and rerun, or install Transformers '
        'from source before running the notebook.'
    )


for module_name, pip_spec in REQUIRED_RUNTIME.items():
    ensure_package(module_name, pip_spec, force_upgrade=FORCE_MODERN_HF_RUNTIME)
ensure_transformers_for_qwen35()

missing_core = [module_name for module_name in CORE_MODULES if not module_exists(module_name)]
if missing_core:
    raise ImportError('Missing core dependencies: ' + ', '.join(missing_core))

import datasets
import torch
import transformers

print(f'Python:       {sys.version.split()[0]}')
print(f'Interpreter:  {sys.executable}')
print(f'Platform:     {platform.platform()}')
print(f'Kaggle:       {IS_KAGGLE}')
print(f'torch:        {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
print(f'transformers: {transformers.__version__}')
print(f'datasets:     {datasets.__version__}')

if IS_KAGGLE and not torch.cuda.is_available():
    print('Warning: Kaggle GPU is not enabled. The notebook can run on CPU, but Qwen inference will be very slow.')


In [ ]:
import dataclasses
import gc
import json
import random
import re
import typing
from collections.abc import Iterable

import numpy as np
import pandas as pd
import sklearn.metrics
import sklearn.model_selection
from IPython.display import display


## Configuration

The comparison follows the homework requirement: zero-shot, few-shot, chain-of-thought-style instruction, and a self-check instruction across Qwen model sizes. The selection rule below uses **validation macro F1** as the primary metric, with accuracy as a tie-breaker. This matches the report and avoids selecting a system that improves raw accuracy by ignoring a minority or boundary class.

If the 4B model does not fit in the active GPU, remove `qwen-4b` from `MODELS_TO_RUN` and rerun the notebook; the selection and submission code remains unchanged.


In [ ]:
RANDOM_STATE = 42
OUTPUT_DIR = Path('runs_hw3_kaggle')
SUBMISSION_FILENAME = 'submission_best_prompting_system.csv'

MODELS = {
    'qwen-0.8b': 'Qwen/Qwen3.5-0.8B',
    'qwen-2b': 'Qwen/Qwen3.5-2B',
    'qwen-4b': 'Qwen/Qwen3.5-4B',
}

MODELS_TO_RUN = ['qwen-0.8b', 'qwen-2b', 'qwen-4b']
STRATEGIES_TO_RUN = ['zero-shot', 'few-shot', 'cot', 'self-check']

VALIDATION_FRACTION = 0.20
VALIDATION_PER_LABEL = 10
K_SHOTS = 3
K_PER_LABEL = 1
FEWSHOT_STRATEGY = 'balanced'

MAX_QUESTION_CHARS = 300
MAX_ANSWER_CHARS = 900
MAX_NEW_TOKENS = 48
BATCH_SIZE = 1
DEVICE_MAP = 'auto'
TORCH_DTYPE = 'auto'
USE_CHAT_TEMPLATE = True
ENABLE_THINKING = False
SYSTEM_MESSAGE = 'You are a careful annotator for political interview response clarity.'

CLARITY_LABELS = ('Clear Reply', 'Ambivalent', 'Clear Non-Reply')


## Data Loading

The official dataset is loaded from Hugging Face. We use a stratified split of the training data for validation, then train/few-shot-select from the remaining training portion during model comparison. For the final submission, the selected prompting system uses the full labeled training split as its demonstration pool.

In [ ]:
def seed_everything(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
    except Exception:
        pass
else:
    try:
        import dotenv
        dotenv.load_dotenv(override=True)
    except Exception:
        pass


def clean_text(value: typing.Any) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split())


def canonical_clarity_label(value: typing.Any) -> str:
    text = clean_text(value)
    normalized = re.sub(r'[-_]+', ' ', text).casefold()
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    if normalized in {'clear reply', 'explicit'}:
        return 'Clear Reply'
    if normalized in {'ambivalent', 'ambivalent reply'}:
        return 'Ambivalent'
    if normalized in {'clear non reply', 'clear not reply', 'clear non-reply', 'clear not-reply'}:
        return 'Clear Non-Reply'
    return text


def canonical_evasion_label(value: typing.Any) -> str:
    text = clean_text(value)
    text = re.sub(r'^\d+(?:\.\d+)*\s*', '', text)
    aliases = {
        'Declining': 'Declining to answer',
        'Ignorance': 'Claims ignorance',
        'Partial': 'Partial/half-answer',
    }
    return aliases.get(text, text)


def normalize_frame(frame: pd.DataFrame, require_label: bool) -> pd.DataFrame:
    required = ['question', 'interview_answer']
    missing = [column for column in required if column not in frame]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    if require_label and 'clarity_label' not in frame:
        raise ValueError('Training data must include clarity_label.')

    result = frame.copy()
    if 'clarity_label' not in result:
        result['clarity_label'] = ''
    if 'evasion_label' not in result:
        result['evasion_label'] = ''

    result['question'] = result['question'].map(clean_text)
    result['interview_answer'] = result['interview_answer'].map(clean_text)
    result['clarity_label'] = result['clarity_label'].map(canonical_clarity_label)
    result['evasion_label'] = result['evasion_label'].map(canonical_evasion_label)
    return result[['question', 'interview_answer', 'clarity_label', 'evasion_label']].fillna('')


seed_everything(RANDOM_STATE)
raw_data = datasets.load_dataset('ailsntua/QEvasion')
train_all = normalize_frame(raw_data['train'].to_pandas(), require_label=True)
test_df = normalize_frame(raw_data['test'].to_pandas(), require_label=False)

train_fit, validation_df = sklearn.model_selection.train_test_split(
    train_all,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    stratify=train_all['clarity_label'],
)
train_fit = train_fit.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train for comparison: {len(train_fit)}')
print(f'Validation split:     {len(validation_df)}')
print(f'Test split:           {len(test_df)}')
display(train_all['clarity_label'].value_counts().rename_axis('label').reset_index(name='count'))


## Prompting and Parsing Toolkit

The prompt uses only canonical output labels: `Clear Reply`, `Ambivalent`, and `Clear Non-Reply`. The dataset's fine-grained evasion labels are included as a bare taxonomy because they define how the three clarity labels are organized.

In [ ]:
@dataclasses.dataclass(frozen=True)
class LabelSpec:
    name: str
    description: str
    aliases: tuple[str, ...] = ()


@dataclasses.dataclass(frozen=True)
class EvasionSpec:
    name: str
    clarity_label: str
    description: str = ''


DEFAULT_LABEL_SPECS = (
    LabelSpec(
        'Clear Reply',
        'The answer explicitly, directly, and specifically addresses the main question.',
        aliases=('direct reply', 'clear answer', 'direct answer', 'explicit reply', 'explicit answer', 'explicit'),
    ),
    LabelSpec(
        'Ambivalent',
        'The answer is partly responsive but incomplete, implicit, vague, hedged, generic, deflective, or only partly answers the question.',
        aliases=('ambivalent reply', 'ambiguous', 'partial reply', 'unclear', 'implicit', 'dodging', 'general', 'deflection', 'partial', 'partial half answer', 'half answer'),
    ),
    LabelSpec(
        'Clear Non-Reply',
        'The answer clearly avoids, redirects, refuses, or otherwise provides no substantive answer to the main question.',
        aliases=('clear not reply', 'non-reply', 'non reply', 'not reply', 'evasion', 'declining', 'declining to answer', 'ignorance', 'claims ignorance', 'clarification'),
    ),
)

EVASION_DESCRIPTIONS = {
    'Explicit': 'Directly and explicitly answers the question.',
    'Implicit': 'Gives an implied or indirect answer, but not a fully explicit one.',
    'Dodging': 'Moves around the question while still touching related material.',
    'General': 'Responds with broad or generic statements instead of the requested specific answer.',
    'Deflection': 'Shifts emphasis toward another issue while keeping some connection to the question.',
    'Partial/half-answer': 'Answers only part of the question or leaves the requested point unresolved.',
    'Declining to answer': 'Openly refuses or declines to answer.',
    'Claims ignorance': 'States lack of knowledge or inability to answer.',
    'Clarification': 'Asks for clarification or challenges the question instead of answering it.',
}

FALLBACK_EVASION_SPECS = (
    EvasionSpec('Explicit', 'Clear Reply', EVASION_DESCRIPTIONS['Explicit']),
    EvasionSpec('Implicit', 'Ambivalent', EVASION_DESCRIPTIONS['Implicit']),
    EvasionSpec('Dodging', 'Ambivalent', EVASION_DESCRIPTIONS['Dodging']),
    EvasionSpec('General', 'Ambivalent', EVASION_DESCRIPTIONS['General']),
    EvasionSpec('Deflection', 'Ambivalent', EVASION_DESCRIPTIONS['Deflection']),
    EvasionSpec('Partial/half-answer', 'Ambivalent', EVASION_DESCRIPTIONS['Partial/half-answer']),
    EvasionSpec('Declining to answer', 'Clear Non-Reply', EVASION_DESCRIPTIONS['Declining to answer']),
    EvasionSpec('Claims ignorance', 'Clear Non-Reply', EVASION_DESCRIPTIONS['Claims ignorance']),
    EvasionSpec('Clarification', 'Clear Non-Reply', EVASION_DESCRIPTIONS['Clarification']),
)


def infer_evasion_specs(source: pd.DataFrame) -> tuple[EvasionSpec, ...]:
    if 'clarity_label' not in source or 'evasion_label' not in source:
        return FALLBACK_EVASION_SPECS
    pairs = set()
    for _, record in source[['clarity_label', 'evasion_label']].fillna('').iterrows():
        label = canonical_clarity_label(record['clarity_label'])
        evasion = canonical_evasion_label(record['evasion_label'])
        if label and evasion:
            pairs.add((label, evasion))
    if not pairs:
        return FALLBACK_EVASION_SPECS
    order = {label: i for i, label in enumerate(CLARITY_LABELS)}
    return tuple(
        EvasionSpec(evasion, label, EVASION_DESCRIPTIONS.get(evasion, 'Dataset subtype.'))
        for label, evasion in sorted(pairs, key=lambda pair: (order.get(pair[0], len(order)), pair[1]))
    )


def truncate_text(text: str, max_chars: int | None) -> str:
    if max_chars is None or len(text) <= max_chars:
        return text
    marker = ' ... '
    available = max_chars - len(marker)
    if available <= 0:
        return text[:max_chars]
    head = max(1, available // 2)
    tail = max(1, available - head)
    return text[:head].rstrip() + marker + text[-tail:].lstrip()


@dataclasses.dataclass(frozen=True)
class PromptExample:
    question: str
    answer: str
    label: str
    evasion_label: str | None = None

    @classmethod
    def from_record(cls, record: typing.Mapping[str, typing.Any]) -> 'PromptExample':
        return cls(
            question=clean_text(record.get('question', '')),
            answer=clean_text(record.get('interview_answer', '')),
            label=canonical_clarity_label(record.get('clarity_label', '')),
            evasion_label=canonical_evasion_label(record.get('evasion_label', '')) or None,
        )


@dataclasses.dataclass(frozen=True)
class PromptConfig:
    name: str
    reasoning: str = 'none'
    include_examples: bool = False


PROMPT_CONFIGS = {
    'zero-shot': PromptConfig('zero-shot', reasoning='none', include_examples=False),
    'few-shot': PromptConfig('few-shot', reasoning='none', include_examples=True),
    'cot': PromptConfig('cot', reasoning='step_by_step', include_examples=True),
    'self-check': PromptConfig('self-check', reasoning='self_check', include_examples=True),
}


class FewShotSampler:
    def __init__(self, k: int = 3, k_per_label: int | None = 1, strategy: str = 'balanced', seed: int = RANDOM_STATE) -> None:
        self.k = k
        self.k_per_label = k_per_label
        self.strategy = strategy
        self.seed = seed
        self.examples: list[PromptExample] = []

    def fit(self, source: pd.DataFrame) -> 'FewShotSampler':
        self.examples = [PromptExample.from_record(record) for _, record in source.iterrows() if clean_text(record.get('clarity_label', ''))]
        return self

    def select(self, record: pd.Series | None = None) -> list[PromptExample]:
        if self.strategy == 'random':
            rng = random.Random(self.seed)
            return rng.sample(self.examples, k=min(self.k, len(self.examples)))

        per_label = self.k_per_label
        if per_label is None:
            per_label = max(1, self.k // len(CLARITY_LABELS))

        selected: list[PromptExample] = []
        rng = random.Random(self.seed)
        for label in CLARITY_LABELS:
            label_examples = [example for example in self.examples if example.label == label]
            if self.strategy == 'length_matched' and record is not None:
                target_len = len(clean_text(record.get('question', '')).split()) + len(clean_text(record.get('interview_answer', '')).split())
                label_examples.sort(key=lambda example: abs(len((example.question + ' ' + example.answer).split()) - target_len))
            else:
                rng.shuffle(label_examples)
            selected.extend(label_examples[:per_label])
        return selected[:self.k] if self.k else selected


class PromptBuilder:
    def __init__(self, config: PromptConfig, evasion_specs: tuple[EvasionSpec, ...]) -> None:
        self.config = config
        self.evasion_specs = evasion_specs

    def build(self, record: pd.Series, examples: list[PromptExample] | None = None) -> str:
        examples = examples or []
        parts = [
            "Classify the interview answer's responsiveness to the question.",
            'Use exactly one of these labels: Clear Reply, Ambivalent, Clear Non-Reply.',
            self._taxonomy_block(),
        ]
        if examples:
            parts.append(self._examples_block(examples))
        parts.append(self._instance_block(record))
        parts.append(self._reasoning_block())
        parts.append(
            'Output constraints:\n'
            '- Your entire response must be exactly one of these labels: Clear Reply | Ambivalent | Clear Non-Reply.\n'
            '- Do not output JSON.\n'
            '- Do not output explanations, markdown, punctuation, or any extra text.'
        )
        return '\n\n'.join(parts)

    def _taxonomy_block(self) -> str:
        lines = ['Label taxonomy (dataset subtypes under each allowed label; output only the allowed label):']
        for label in CLARITY_LABELS:
            subtypes = [spec.name for spec in self.evasion_specs if spec.clarity_label == label]
            if subtypes:
                lines.append(f'- {label}: {", ".join(subtypes)}')
        return '\n'.join(lines)

    def _examples_block(self, examples: list[PromptExample]) -> str:
        lines = ['Examples:']
        for i, example in enumerate(examples, start=1):
            lines.extend([
                f'Example {i}:',
                f'Question: {truncate_text(example.question, MAX_QUESTION_CHARS)}',
                f'Answer: {truncate_text(example.answer, MAX_ANSWER_CHARS)}',
                f'Label: {example.label}',
                '',
            ])
        while lines and lines[-1] == '':
            lines.pop()
        return '\n'.join(lines)

    def _instance_block(self, record: pd.Series) -> str:
        question = truncate_text(clean_text(record.get('question', '')), MAX_QUESTION_CHARS)
        answer = truncate_text(clean_text(record.get('interview_answer', '')), MAX_ANSWER_CHARS)
        return f'Instance to classify:\nQuestion: {question}\nAnswer: {answer}'

    def _reasoning_block(self) -> str:
        if self.config.reasoning == 'step_by_step':
            return 'Think through whether the response addresses the asked content, whether it is specific or vague, and whether it redirects away from the question. Output only the final label.'
        if self.config.reasoning == 'self_check':
            return 'Choose the most likely label, check that it is one of the allowed labels and that the answer is judged relative to the question, then output only the final label.'
        return "Decide the label from the answer's responsiveness to the question. Do not output explanations."


class PromptEncoder:
    def __init__(self, strategy: str, evasion_specs: tuple[EvasionSpec, ...]) -> None:
        self.strategy = strategy
        self.config = PROMPT_CONFIGS[strategy]
        self.builder = PromptBuilder(self.config, evasion_specs)
        self.sampler = FewShotSampler(K_SHOTS, K_PER_LABEL, FEWSHOT_STRATEGY) if self.config.include_examples else None

    def fit(self, source: pd.DataFrame) -> 'PromptEncoder':
        if self.sampler is not None:
            self.sampler.fit(source)
        return self

    def transform(self, source: pd.DataFrame) -> list[str]:
        prompts = []
        for _, record in source.iterrows():
            examples = self.sampler.select(record) if self.sampler is not None else []
            prompts.append(self.builder.build(record, examples))
        return prompts


In [ ]:
LABEL_FIELD_NAMES = ('label', 'final_label', 'predicted_label', 'prediction', 'class', 'category')


def normalization_key(text: str) -> str:
    text = text.casefold()
    text = re.sub(r'[`"\'*_{}\[\]().,:;!?]', ' ', text)
    text = re.sub(r'[/_]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def strip_label_noise(text: str) -> str:
    text = re.sub(r'^(the\s+)?(final\s+)?(label|answer|prediction|class|category)\s+(is|=|:)?\s+', '', text)
    text = re.sub(r'\s+(because|since|as)\s+.*$', '', text)
    return text.strip()


class LabelNormalizer:
    def __init__(self, labels: tuple[LabelSpec, ...] = DEFAULT_LABEL_SPECS) -> None:
        self.alias_to_label: dict[str, str] = {}
        for label in labels:
            self._register(label.name, label.name)
            self._register(label.name.replace('-', ' '), label.name)
            self._register(label.name.replace('-', ''), label.name)
            for alias in label.aliases:
                self._register(alias, label.name)

    def _register(self, alias: str, label: str) -> None:
        key = normalization_key(alias)
        if key:
            self.alias_to_label[key] = label

    def normalize(self, value: typing.Any) -> str | None:
        text = normalization_key(str(value))
        if text in self.alias_to_label:
            return self.alias_to_label[text]
        text = strip_label_noise(text)
        return self.alias_to_label.get(text)

    def find_mentions(self, text: str) -> list[tuple[int, str]]:
        mentions = []
        normalized = normalization_key(text)
        for alias_key, label in self.alias_to_label.items():
            pattern = r'(?<!\w)' + re.escape(alias_key).replace(r'\ ', r'[\s-]+') + r'(?!\w)'
            for match in re.finditer(pattern, normalized):
                mentions.append((match.start(), label))
        return sorted(mentions, key=lambda item: item[0])


@dataclasses.dataclass(frozen=True)
class ParsedGeneration:
    raw_text: str
    label: str | None
    valid: bool
    parse_method: str
    message: str = ''


class GenerationParser:
    def __init__(self, default_label: str | None = None) -> None:
        self.normalizer = LabelNormalizer()
        self.default_label = default_label

    def parse(self, text: typing.Any) -> ParsedGeneration:
        raw_text = '' if text is None else str(text).strip()
        for parsed_object in reversed(list(self._iter_json_objects(raw_text))):
            if isinstance(parsed_object, dict):
                for key in LABEL_FIELD_NAMES:
                    if key in parsed_object:
                        label = self.normalizer.normalize(parsed_object[key])
                        if label is not None:
                            return ParsedGeneration(raw_text, label, True, 'json')

        pattern = re.compile(r'(?im)^\s*(?:final\s+)?(?:label|answer|prediction|class|category)\s*[:=-]\s*(.+?)\s*$')
        for match in reversed(list(pattern.finditer(raw_text))):
            label = self.normalizer.normalize(match.group(1))
            if label is not None:
                return ParsedGeneration(raw_text, label, True, 'labeled_line')

        mentions = self.normalizer.find_mentions(raw_text)
        if mentions:
            unique_labels = {label for _, label in mentions}
            if len(unique_labels) == 1:
                return ParsedGeneration(raw_text, next(iter(unique_labels)), True, 'single_mention')
            last_position, last_label = mentions[-1]
            window = normalization_key(raw_text)[max(0, last_position - 40):last_position + 120]
            if re.search(r'\b(final|therefore|answer|label|prediction)\b', window):
                return ParsedGeneration(raw_text, last_label, True, 'final_mention')
            return ParsedGeneration(raw_text, None, False, 'ambiguous_mentions', f'Multiple labels mentioned: {sorted(unique_labels)}')

        if self.default_label is not None:
            return ParsedGeneration(raw_text, self.default_label, False, 'default', 'No valid label found; default used.')
        return ParsedGeneration(raw_text, None, False, 'invalid', 'No valid clarity label found.')

    def parse_many(self, texts: Iterable[typing.Any]) -> list[ParsedGeneration]:
        return [self.parse(text) for text in texts]

    def _iter_json_objects(self, text: str):
        decoder = json.JSONDecoder()
        for start in [match.start() for match in re.finditer(r'\{', text)]:
            try:
                parsed, _ = decoder.raw_decode(text[start:])
            except json.JSONDecodeError:
                continue
            yield parsed


## Hugging Face Generation Backend

The generation wrapper uses deterministic decoding (`do_sample=False`) and asks the model for a short continuation containing only the label. It supports current Qwen chat templates and falls back between the `AutoModelForImageTextToText` and causal language model APIs when needed.

In [ ]:
@dataclasses.dataclass(frozen=True)
class DecodingConfig:
    max_new_tokens: int = MAX_NEW_TOKENS
    do_sample: bool = False
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.0


class HuggingFaceGenerator:
    def __init__(
        self,
        model_name: str,
        decoding: DecodingConfig | None = None,
        batch_size: int = BATCH_SIZE,
        device_map: str | dict[str, typing.Any] | None = DEVICE_MAP,
        torch_dtype: str | typing.Any = TORCH_DTYPE,
        use_chat_template: bool = USE_CHAT_TEMPLATE,
        system_message: str | None = SYSTEM_MESSAGE,
        enable_thinking: bool | None = ENABLE_THINKING,
    ) -> None:
        self.model_name = model_name
        self.decoding = decoding or DecodingConfig()
        self.batch_size = batch_size
        self.device_map = device_map
        self.torch_dtype = torch_dtype
        self.use_chat_template = use_chat_template
        self.system_message = system_message
        self.enable_thinking = enable_thinking
        self._tokenizer = None
        self._processor = None
        self._model = None
        self._loaded_backend = None

    def generate(self, prompts: list[str]) -> list[str]:
        self._load()
        if self._loaded_backend == 'image-text-to-text':
            return self._generate_image_text_to_text(prompts)
        return self._generate_causal_lm(prompts)

    def _load(self) -> None:
        if self._model is not None:
            return
        if hasattr(transformers, 'AutoModelForImageTextToText'):
            try:
                self._processor = transformers.AutoProcessor.from_pretrained(self.model_name, trust_remote_code=True)
                self._ensure_processor_padding_token()
                model_cls = getattr(transformers, 'AutoModelForImageTextToText')
                self._model = model_cls.from_pretrained(
                    self.model_name,
                    device_map=self.device_map,
                    torch_dtype=self.torch_dtype,
                    trust_remote_code=True,
                )
                self._sync_generation_config_token_ids()
                self._model.eval()
                self._loaded_backend = 'image-text-to-text'
                return
            except Exception as error:
                print(f'Image-text-to-text load failed for {self.model_name}: {error}')
                self._processor = None
                self._model = None

        self._tokenizer = transformers.AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        if self._tokenizer.pad_token_id is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token
        self._model = transformers.AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map=self.device_map,
            torch_dtype=self.torch_dtype,
            trust_remote_code=True,
        )
        self._model.eval()
        self._loaded_backend = 'causal-lm'

    def _generate_causal_lm(self, prompts: list[str]) -> list[str]:
        outputs = []
        for start in range(0, len(prompts), self.batch_size):
            batch_prompts = [self._format_prompt(prompt) for prompt in prompts[start:start + self.batch_size]]
            encoded = self._tokenizer(batch_prompts, return_tensors='pt', padding=True)
            encoded = self._move_batch(encoded)
            input_width = encoded['input_ids'].shape[1]
            with torch.no_grad():
                generated = self._model.generate(
                    **encoded,
                    pad_token_id=self._tokenizer.pad_token_id,
                    eos_token_id=self._tokenizer.eos_token_id,
                    **self._generation_kwargs(),
                )
            for sequence in generated:
                continuation = sequence[input_width:]
                outputs.append(self._tokenizer.decode(continuation, skip_special_tokens=True).strip())
        return outputs

    def _generate_image_text_to_text(self, prompts: list[str]) -> list[str]:
        outputs = []
        for start in range(0, len(prompts), self.batch_size):
            messages = [self._messages(prompt, structured_content=True) for prompt in prompts[start:start + self.batch_size]]
            batch_text = [
                self._apply_chat_template(self._processor, message, tokenize=False, add_generation_prompt=True)
                for message in messages
            ]
            encoded = self._processor(text=batch_text, return_tensors='pt', padding=True)
            encoded = self._move_batch(encoded)
            input_width = encoded['input_ids'].shape[1]
            with torch.no_grad():
                generated = self._model.generate(
                    **encoded,
                    **self._generation_token_kwargs(),
                    **self._generation_kwargs(),
                )
            continuations = generated[:, input_width:]
            outputs.extend(text.strip() for text in self._processor.batch_decode(continuations, skip_special_tokens=True))
        return outputs

    def _format_prompt(self, prompt: str) -> str:
        if not self.use_chat_template or self._tokenizer is None or not getattr(self._tokenizer, 'chat_template', None):
            return prompt
        return self._apply_chat_template(
            self._tokenizer,
            self._messages(prompt, structured_content=False),
            tokenize=False,
            add_generation_prompt=True,
        )

    def _messages(self, prompt: str, structured_content: bool) -> list[dict[str, typing.Any]]:
        messages = []
        if self.system_message:
            content = [{'type': 'text', 'text': self.system_message}] if structured_content else self.system_message
            messages.append({'role': 'system', 'content': content})
        content = [{'type': 'text', 'text': prompt}] if structured_content else prompt
        messages.append({'role': 'user', 'content': content})
        return messages

    def _apply_chat_template(self, owner: typing.Any, messages: list[dict[str, typing.Any]], **kwargs: typing.Any) -> typing.Any:
        if self.enable_thinking is not None:
            kwargs['enable_thinking'] = self.enable_thinking
        try:
            return owner.apply_chat_template(messages, **kwargs)
        except TypeError as error:
            if 'enable_thinking' not in kwargs:
                raise
            if 'enable_thinking' not in str(error) and 'unexpected keyword' not in str(error):
                raise
            kwargs.pop('enable_thinking')
            return owner.apply_chat_template(messages, **kwargs)

    def _generation_kwargs(self) -> dict[str, typing.Any]:
        kwargs = {
            'max_new_tokens': self.decoding.max_new_tokens,
            'do_sample': self.decoding.do_sample,
            'repetition_penalty': self.decoding.repetition_penalty,
        }
        if self.decoding.do_sample:
            kwargs['temperature'] = self.decoding.temperature
            kwargs['top_p'] = self.decoding.top_p
        return kwargs

    def _generation_token_kwargs(self) -> dict[str, typing.Any]:
        eos_token_id = self._token_id('eos_token_id')
        pad_token_id = self._token_id('pad_token_id') or eos_token_id
        kwargs = {}
        if pad_token_id is not None:
            kwargs['pad_token_id'] = pad_token_id
        if eos_token_id is not None:
            kwargs['eos_token_id'] = eos_token_id
        return kwargs

    def _ensure_processor_padding_token(self) -> None:
        tokenizer = getattr(self._processor, 'tokenizer', None)
        if tokenizer is not None and getattr(tokenizer, 'pad_token_id', None) is None and getattr(tokenizer, 'eos_token', None) is not None:
            tokenizer.pad_token = tokenizer.eos_token

    def _sync_generation_config_token_ids(self) -> None:
        if self._model is None:
            return
        config = getattr(self._model, 'generation_config', None)
        if config is None:
            return
        eos_token_id = self._token_id('eos_token_id')
        pad_token_id = self._token_id('pad_token_id') or eos_token_id
        if getattr(config, 'pad_token_id', None) is None and pad_token_id is not None:
            config.pad_token_id = pad_token_id
        if getattr(config, 'eos_token_id', None) is None and eos_token_id is not None:
            config.eos_token_id = eos_token_id

    def _token_id(self, name: str) -> typing.Any:
        tokenizer = self._tokenizer or getattr(self._processor, 'tokenizer', None)
        value = getattr(tokenizer, name, None) if tokenizer is not None else None
        if value is not None:
            return value
        config = getattr(self._model, 'generation_config', None) if self._model is not None else None
        return getattr(config, name, None)

    def _move_batch(self, batch: typing.Any) -> typing.Any:
        device = getattr(self._model, 'device', None)
        if hasattr(batch, 'to') and device is not None:
            return batch.to(device)
        if device is None:
            return batch
        return {key: value.to(device) if hasattr(value, 'to') else value for key, value in batch.items()}


class PromptedGenerationClassifier:
    def __init__(self, strategy: str, model_name: str, evasion_specs: tuple[EvasionSpec, ...]) -> None:
        self.encoder = PromptEncoder(strategy, evasion_specs)
        self.generator = HuggingFaceGenerator(model_name)
        self.parser = GenerationParser()

    def fit(self, source: pd.DataFrame) -> 'PromptedGenerationClassifier':
        self.encoder.fit(source)
        return self

    def generate_frame(self, source: pd.DataFrame, include_prompts: bool = True) -> pd.DataFrame:
        prompts = self.encoder.transform(source)
        generations = self.generator.generate(prompts)
        parsed = self.parser.parse_many(generations)
        frame = pd.DataFrame({
            'generation': generations,
            'Predicted': [item.label for item in parsed],
            'valid': [item.valid for item in parsed],
            'parse_method': [item.parse_method for item in parsed],
            'parse_message': [item.message for item in parsed],
        }, index=source.index)
        frame.index.name = source.index.name or 'Id'
        if include_prompts:
            frame.insert(0, 'prompt', prompts)
        return frame


## Evaluation Helpers

Invalid generations are counted explicitly in validation metrics. For the final Kaggle file, any invalid prediction is mapped to `Ambivalent` so every row has a legal label.

In [ ]:
def classification_scores(true: Iterable[str], pred: Iterable[str | None]) -> dict[str, float]:
    y_true = pd.Series(list(true), dtype='object')
    y_pred = pd.Series(list(pred), dtype='object')
    valid = y_pred.isin(CLARITY_LABELS)
    y_eval = y_pred.where(valid, '__invalid__')
    return {
        'accuracy': float(sklearn.metrics.accuracy_score(y_true, y_eval)),
        'precision_macro': float(sklearn.metrics.precision_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='macro', zero_division=0)),
        'recall_macro': float(sklearn.metrics.recall_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='macro', zero_division=0)),
        'f1_macro': float(sklearn.metrics.f1_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='macro', zero_division=0)),
        'f1_weighted': float(sklearn.metrics.f1_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='weighted', zero_division=0)),
        'f1_micro': float(sklearn.metrics.f1_score(y_true, y_eval, labels=list(CLARITY_LABELS), average='micro', zero_division=0)),
        'invalid_rate': float((~valid).mean()) if len(valid) else 0.0,
    }


def confusion_matrix_frame(true: Iterable[str], pred: Iterable[str | None]) -> pd.DataFrame:
    y_true = pd.Series(list(true), dtype='object')
    y_pred = pd.Series(list(pred), dtype='object')
    labels = list(CLARITY_LABELS)
    if (~y_pred.isin(CLARITY_LABELS)).any():
        y_pred = y_pred.where(y_pred.isin(CLARITY_LABELS), '__invalid__')
        labels.append('__invalid__')
    matrix = sklearn.metrics.confusion_matrix(y_true, y_pred, labels=labels)
    return pd.DataFrame(matrix, index=labels, columns=labels)


def classification_report_frame(true: Iterable[str], pred: Iterable[str | None]) -> pd.DataFrame:
    y_true = pd.Series(list(true), dtype='object')
    y_pred = pd.Series(list(pred), dtype='object')
    y_eval = y_pred.where(y_pred.isin(CLARITY_LABELS), '__invalid__')
    return pd.DataFrame(
        sklearn.metrics.classification_report(
            y_true,
            y_eval,
            labels=list(CLARITY_LABELS),
            output_dict=True,
            zero_division=0,
        )
    ).transpose()


def balanced_label_sample(frame: pd.DataFrame, per_label: int, seed: int = RANDOM_STATE) -> pd.DataFrame:
    if per_label <= 0:
        return frame.reset_index(drop=True)
    parts = []
    for i, label in enumerate(CLARITY_LABELS):
        group = frame[frame['clarity_label'] == label]
        if len(group) < per_label:
            raise ValueError(f'Cannot sample {per_label} examples for {label}; only {len(group)} available.')
        parts.append(group.sample(n=per_label, random_state=seed + i))
    return pd.concat(parts).sort_index().reset_index(drop=True)


def make_length_bucket(series: pd.Series) -> pd.Series:
    q1, q2 = series.quantile([0.33, 0.67])
    return pd.cut(series, bins=[-np.inf, q1, q2, np.inf], labels=['short', 'medium', 'long'], include_lowest=True)


def length_subgroup_scores(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result['question_words'] = result['question'].str.split().str.len()
    result['answer_words'] = result['interview_answer'].str.split().str.len()
    rows = []
    for length_col in ['question_words', 'answer_words']:
        bucket_col = f'{length_col}_bucket'
        result[bucket_col] = make_length_bucket(result[length_col])
        for group, group_frame in result.groupby(bucket_col, observed=True):
            row = {'group_col': bucket_col, 'group': str(group), 'n': len(group_frame)}
            row.update(classification_scores(group_frame['clarity_label'], group_frame['Predicted']))
            rows.append(row)
    return pd.DataFrame(rows)


def submission_frame(predictions: Iterable[str | None]) -> pd.DataFrame:
    values = [prediction if prediction in CLARITY_LABELS else 'Ambivalent' for prediction in predictions]
    return pd.DataFrame({'Id': range(len(values)), 'Predicted': values})


## Validation Comparison

The following cell runs the model/strategy grid, saves every validation generation, and builds a summary table. The final selection is the row with the highest validation macro F1; accuracy breaks ties.


In [ ]:
def run_id(model_key: str, strategy: str) -> str:
    return f'{model_key}_{strategy}'


def run_experiment(
    model_key: str,
    strategy: str,
    train_source: pd.DataFrame,
    evaluation_source: pd.DataFrame,
    evasion_specs: tuple[EvasionSpec, ...],
) -> tuple[dict[str, typing.Any], pd.DataFrame]:
    model_name = MODELS[model_key]
    classifier = PromptedGenerationClassifier(strategy, model_name, evasion_specs).fit(train_source)
    generated = classifier.generate_frame(evaluation_source, include_prompts=True)
    joined = pd.concat([evaluation_source.reset_index(drop=True), generated.reset_index(drop=True)], axis=1)
    scores = classification_scores(joined['clarity_label'], joined['Predicted'])

    row: dict[str, typing.Any] = {
        'model_key': model_key,
        'model': model_name,
        'strategy': strategy,
        'run_id': run_id(model_key, strategy),
        'n_validation': len(joined),
        'prompt_chars_mean': float(joined['prompt'].str.len().mean()),
        'prompt_chars_max': int(joined['prompt'].str.len().max()),
    }
    row.update(scores)
    for label in CLARITY_LABELS:
        row[f'truth_{label}'] = int((joined['clarity_label'] == label).sum())
        row[f'pred_{label}'] = int((joined['Predicted'] == label).sum())
    row['valid_rate'] = float(joined['valid'].mean())
    return row, joined


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'runs').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'tables').mkdir(parents=True, exist_ok=True)

evaluation_df = balanced_label_sample(validation_df, VALIDATION_PER_LABEL, seed=RANDOM_STATE)
evasion_specs = infer_evasion_specs(train_all)
rows = []
run_frames = {}

for model_key in MODELS_TO_RUN:
    for strategy in STRATEGIES_TO_RUN:
        print('=' * 80)
        print(f'MODEL:    {model_key} -> {MODELS[model_key]}')
        print(f'STRATEGY: {strategy}')
        try:
            row, frame = run_experiment(model_key, strategy, train_fit, evaluation_df, evasion_specs)
        except Exception as error:
            print(f'Run failed and will be skipped during selection: {type(error).__name__}: {error}')
            row = {
                'model_key': model_key,
                'model': MODELS[model_key],
                'strategy': strategy,
                'run_id': run_id(model_key, strategy),
                'n_validation': len(evaluation_df),
                'accuracy': np.nan,
                'precision_macro': np.nan,
                'recall_macro': np.nan,
                'f1_macro': np.nan,
                'f1_weighted': np.nan,
                'f1_micro': np.nan,
                'invalid_rate': np.nan,
                'valid_rate': np.nan,
                'failed': True,
                'error': repr(error),
            }
            rows.append(row)
        else:
            row['failed'] = False
            rows.append(row)
            run_name = row['run_id']
            run_frames[run_name] = frame
            frame.to_csv(OUTPUT_DIR / 'runs' / f'{run_name}.generations.csv', index=False)
            classification_report_frame(frame['clarity_label'], frame['Predicted']).to_csv(OUTPUT_DIR / 'runs' / f'{run_name}.classification_report.csv')
            confusion_matrix_frame(frame['clarity_label'], frame['Predicted']).to_csv(OUTPUT_DIR / 'runs' / f'{run_name}.confusion.csv')
            length_subgroup_scores(frame).to_csv(OUTPUT_DIR / 'runs' / f'{run_name}.length_subgroups.csv', index=False)
            print({key: row[key] for key in ['accuracy', 'f1_macro', 'invalid_rate', 'valid_rate']})
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

results_df = pd.DataFrame(rows).sort_values(['f1_macro', 'accuracy'], ascending=[False, False], na_position='last').reset_index(drop=True)
if results_df['accuracy'].notna().sum() == 0:
    raise RuntimeError('All validation runs failed; no prompting system can be selected.')
results_df.to_csv(OUTPUT_DIR / 'experiment_summary.csv', index=False)
display(results_df[['model_key', 'strategy', 'f1_macro', 'accuracy', 'precision_macro', 'recall_macro', 'invalid_rate']].round(4))


## Best System Selection

The next cell extracts the best validation run by macro F1, records it, and shows the confusion matrix and per-class report. This is the configuration used for the final Kaggle submission.


In [ ]:
best_row = results_df.iloc[0].to_dict()
best_model_key = str(best_row['model_key'])
best_strategy = str(best_row['strategy'])
best_run_id = str(best_row['run_id'])

with open(OUTPUT_DIR / 'best_validation_system.json', 'w', encoding='utf-8') as file:
    json.dump(best_row, file, indent=2)

print('Selected best prompting system by validation macro F1:')
print(f'Model:    {best_model_key} -> {MODELS[best_model_key]}')
print(f'Strategy: {best_strategy}')
print(f'Accuracy: {best_row["accuracy"]:.4f}')
print(f'Macro F1: {best_row["f1_macro"]:.4f}')

best_frame = run_frames[best_run_id]
print('\nConfusion matrix:')
display(confusion_matrix_frame(best_frame['clarity_label'], best_frame['Predicted']))
print('\nClassification report:')
display(classification_report_frame(best_frame['clarity_label'], best_frame['Predicted']).round(4))
print('\nPrediction distribution:')
display(best_frame['Predicted'].value_counts(dropna=False).rename_axis('Predicted').reset_index(name='count'))


## Error and Subgroup Analysis

The validation sample is small, but it is balanced. These diagnostics make the main failure mode visible before producing the final test submission.

In [ ]:
analysis_df = best_frame.copy()
analysis_df['correct'] = analysis_df['clarity_label'] == analysis_df['Predicted']
errors = analysis_df.loc[~analysis_df['correct']].copy()
errors['confusion'] = errors['clarity_label'] + ' -> ' + errors['Predicted'].fillna('__invalid__')

print(f'Errors: {len(errors)} / {len(analysis_df)}')
display(errors['confusion'].value_counts().rename_axis('confusion').reset_index(name='count'))
display(errors[['clarity_label', 'Predicted', 'question', 'interview_answer']].head(8))

print('Length subgroup scores for selected system:')
display(length_subgroup_scores(analysis_df).round(4))


## Final Kaggle Submission

The selected model/strategy is rerun on the full official test split. The output file is written with the required filename:

```text
submission_best_prompting_system.csv
```

The file has exactly the sample-submission shape: `Id,Predicted`.

In [ ]:
full_train = train_all.reset_index(drop=True)
final_classifier = PromptedGenerationClassifier(best_strategy, MODELS[best_model_key], evasion_specs).fit(full_train)
final_generations = final_classifier.generate_frame(test_df, include_prompts=True)
final_generations.to_csv(OUTPUT_DIR / 'best_prompting_system_generations.csv', index=True)

submission = submission_frame(final_generations['Predicted'])
submission.to_csv(SUBMISSION_FILENAME, index=False)

print(f'Saved Kaggle submission: {SUBMISSION_FILENAME}')
print(f'Rows: {len(submission)}')
print('Prediction distribution:')
display(submission['Predicted'].value_counts().rename_axis('Predicted').reset_index(name='count'))
display(submission.head(10))


## Final Notes

The notebook output is determined by the validation selection rule above. In the accompanying report, the important conclusions to discuss are:

- whether larger Qwen models actually improve the balanced validation sample;
- which prompting method wins under the macro-F1 selection rule;
- whether `Ambivalent` remains the hardest class;
- how often output parsing fails, if at all;
- how these prompting failures compare with the classical and encoder-only systems from HW1/HW2.
